In [ ]:
from IPython.display import HTML, display
display(HTML("""<script type="module">
import mermaid from "https://cdn.jsdelivr.net/npm/mermaid@11/dist/mermaid.esm.min.mjs";
mermaid.initialize({startOnLoad:false, theme:"neutral", securityLevel:"strict"});
await mermaid.run({nodes:document.querySelectorAll(".mermaid:not([data-processed])")});
</script>"""))


# 04c — LangGraph: remediation approval workflow

## Scenario

The incident investigator can prepare a rollback plan but may not restart checkout without a named human. The run must branch, pause, and resume safely after a crash. This is a state-graph problem: graph state is inspectable, conditional routing is explicit, and approval is an interrupt rather than a prompt suggestion.

<pre class="mermaid">
stateDiagram-v2
  [*] --> CollectEvidence
  CollectEvidence --> AssessRisk
  AssessRisk --> PrepareRollback: supported
  AssessRisk --> Escalate: insufficient evidence
  PrepareRollback --> AwaitApproval: restart proposed
  AwaitApproval --> Execute: approved
  AwaitApproval --> Rejected: rejected
  Execute --> [*]
  Escalate --> [*]
  Rejected --> [*]
</pre>


## 1. State before nodes

A production state contract should hold evidence IDs, risk, proposal, authenticated approval, and an idempotency key. Do not make an opaque model response the only record of what happened.

[LangGraph](https://docs.langchain.com/oss/python/langgraph/overview) is useful when state transitions, persistence, retries, and human interrupts are part of the product. It is excessive for a one-step helper function.


In [ ]:
from pathlib import Path
import sys
repo_root = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / "curriculum" / "beginner" / "04-agent-development-frameworks" / "lab.py").exists())
sys.path.insert(0, str(repo_root / "curriculum" / "beginner" / "04-agent-development-frameworks"))
from lab import *


In [ ]:
proposal = langgraph_shaped_approval("prepare_rollback")
paused = langgraph_shaped_approval("restart_checkout")
approved = langgraph_shaped_approval("restart_checkout", approved=True)
print(proposal)
print(paused)
print(approved)
assert paused["state"] == "interrupt"
assert approved["state"] == "complete"


## 2. Approval is not a prompt

A prompt request to “wait for approval” cannot authenticate the approver, bind approval to exact arguments, survive a restart, or prevent duplicate execution. Store an approval record containing:

- approver identity and role;
- normalized action name and arguments;
- evidence digest and risk classification;
- expiration time and single-use idempotency key;
- approve, modify, or reject decision; and
- audit correlation ID.

Validate it at the side-effect boundary. See the official [persistence](https://docs.langchain.com/oss/python/langgraph/persistence) and [interrupts](https://docs.langchain.com/oss/python/langgraph/interrupts) guides.


## 3. Optional real graph shape

```python
from typing import TypedDict
from langgraph.graph import START, END, StateGraph
from langgraph.types import interrupt

class IncidentState(TypedDict):
    evidence_ids: list[str]
    proposal: str | None
    approval: dict | None

def assess(state):
    return {"proposal": "restart_checkout"}

def approval_gate(state):
    decision = interrupt({"action": state["proposal"], "evidence_ids": state["evidence_ids"]})
    return {"approval": decision}

builder = StateGraph(IncidentState)
builder.add_node("assess", assess)
builder.add_node("approval_gate", approval_gate)
builder.add_edge(START, "assess")
builder.add_edge("assess", "approval_gate")
builder.add_edge("approval_gate", END)
graph = builder.compile(checkpointer=...)
```

Use the current documentation for exact APIs. Persist before interruption and validate the trusted resume payload.


In [ ]:
executed = {}
def execute_once(key: str, action: str) -> str:
    if key in executed:
        return f"reused:{executed[key]}"
    executed[key] = action
    return f"executed:{action}"

assert execute_once("approval-482-v1", "restart_checkout").startswith("executed:")
assert execute_once("approval-482-v1", "restart_checkout").startswith("reused:")
print("Idempotency protects replay after a worker retry.")


## 4. Exercises and takeaway

1. Add an evidence-insufficient route that escalates without proposing an action.
2. Expire approval after 15 minutes and test rejection.
3. Require an approver distinct from the operator running the agent.
4. Run a replay test with the same idempotency key.
5. Measure path length, approval wait time, and duplicate-side-effect rate.

**Choose LangGraph** when durable state, explicit branches, and approval/recovery behavior must be reviewable. The graph gives control-flow clarity; it does not replace authorization or idempotent actions.
